# Intro to CometMirror
This notebook is all about running and configuring CometMirror. In this notebook, you'll learn how to:

1. Load up a pre-defined scenario and run a simulation.
2. Specify custom time-dependent trajectories for simulations (e.g. current ramps, impurity injections)
3. Save simulation results as Xarray or Pandas dataframes.
4. Use provided visualization tools
5. Generate random walks for a subset of the parameters, and inject them into the simulation
6. Configure batches of simulation runs using `MultiCases` and `CombinatorialCases`

Let's begin by loading CometMirror for the SPARC PRD and print out the initial `State` and `Params`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pprint import pprint

from popsim.simulate import CombinatorialCases, MultiCases, make_time_base
from popsim.simulators.comet_mirror.scenarios.sparc_prd import build_comet_mirror_config

# Initialize the simulator.
model, state, params = build_comet_mirror_config()

# Make a time base for all of our simulations.
time_base = make_time_base(t0=0.0, t1=5.0, dt=0.01)

pprint(state)
pprint(params)

## Simulating with State + Params

The `State` dataclass is a vector of variables that are being simulated, while `Params` can be thought of as boundary conditions and/or assumptions. Letting $\mathbf{x}_t$ denote the state at time $t$ and $\mathbf{p}_t$ denote the params at time $t$, at every time step of the simulation, what essentially happens is something like this:

$$
\mathbf{x}_{t+1} = f(\mathbf{x}_t, \mathbf{p}_t)
$$

So, in this API, the `model` is sorta like the $f$ function, the `time_base` is the set of time steps we wish to simulate for, the `initial_state` is the initial state $\mathbf{x}_0$, and the `params` are $\mathbf{p}_t$, where some elements can be static and others can be time-dependent.

For a more detailed explanation, checkout the [tutorial on modules](./intro_to_modules.ipynb)

## Specifying Time-Dependent Parameters
The `Params` dataclass has some magical properties. Every element of it is configurable by you, and you can specify time-dependent trajectories using a dictionary mapping time to values. Let's begin by defining a simulation time-base and defining a current ramp that starts 1 second into the simulation. While we're at it, why not define a auxiliary heating ramp rate, and also a tungsten impurity injection that occurs between (2.0, 2.1) seconds in the simulaiton?

In [ ]:
from copy import deepcopy

import jax.numpy as jnp

from popsim.enums import Impurity
from popsim.simulate import SimInput, simulate

# Create a copy of "params". It's best to keep the original one around untouched.
new_params = deepcopy(params)

# Manually define a current-ramp where the key is the time in seconds and the value is the current in Amperes.
new_params.plasma_current = {0.0: 8.7e6, 1.0: 8.7e6, 5.0: 4.0e6}

# Manually define an auxiliary heating power ramp where the key is the time in seconds and the value is the power in MW.
new_params.P_aux_MW = {0.0: 11.1, 5.0: 7.0}

# Manually define a quick tungsten spike. Note that under the hood linear interpolation is happening, so we need this
# perhaps somewhat awkward definition.
new_params.fueling19[Impurity.Tungsten] = {
    0.0: 0.0,
    1.99: 0.0,  # Start ramping impurities.
    2.0: 0.1,  # Impurity injection.
    2.1: 0.1,  # Impurity injection holding.
    2.11: 0.0,  # Impurity drops back to 0.0.
    5.0: 0.0,  # Impurity holds at 0.0.
}


# Build the simulation inputs.
sim_inputs = SimInput(time=time_base, initial_state=state, params=new_params)

dataset = simulate(
    module=model,
    sim_inputs=sim_inputs,
)

## Visualizing Simulation Results
Okay, so we ran a simulation and got a `dataset` object out. By default, this will be an `xarray` dataset, which is a really powerful and useful data analysis package create by geoscientists to help their spatial data problems.

We can view the structure of the data in a Jupyter Notebook by executing the cell below. You'll note that there are **a lot** of variables. This is because `CometMirror` is set up to automatically log all of the variables in the scope of its main body (which if you ask me is pretty cool). You'll notice that the state variables start with `state.`, param variables start with `params.`, and a whole bunch of auxiliary variables are under `output.`.

In [ ]:
dataset

Now, one can extract the variables you care about and write your own plotting functions, but I've also included a generic plotting function `visualize_time_series` that should make your life a bit easier. Just grab the list of variables you care about plotting like what is shown below, and pass it to the function.

In [ ]:
import holoviews as hv

from popsim.visualize import visualize_time_series

hv.extension("matplotlib")

visualize_vars = [
    "output.aux_data.params.plasma_current",
    "output.aux_data.params.P_aux_MW",
    "output.aux_data.params.fueling19.Impurity.Tungsten",
    "state.stored_energy",
    "state.density_state.vol_avg_ion.Impurity.Tungsten",
    "state.hmode_state.hmode",
    "output.aux_data.Prad_imp_MW",
    "output.aux_data.P_rad_MW",
    "output.aux_data.tau_E",
    "output.aux_data.average_electron_density_19",
]
visualize_time_series(dataset[visualize_vars])

## Saving Simulation Runs
The `simulate` function outputs the simulation data as a `xr.Dataset`, which you can save to disk as a `*.h5` or `*.nc` file (`*.nc` is just a special kind of `*.h5` file). Alternatively, you can convert the `xr.Dataset` to a `pd.DataFrame` and save it in whatever pandas format you'd like (e.g. `*.csv`, `*.parquet`, etc). The code below shows an example, where it saves to a temporary directory.

**Best Practice:** `xr.Dataset` is the preferred format. If you convert to pandas, it will be a bit of a headache handling profile variables and multi-simulation datasets.

**Soapbox:** Saving simulation runs as data is useful in certain cases, e.g. to enable interfacing with other codes and analysis suites, but if you want to share your simulation results, it would be better to just share a notebook + git commit, as that will allow others to reproduce your results and increases traceability.

In [ ]:
import os
import tempfile

with tempfile.TemporaryDirectory() as temp_dir:
    # Save the dataset to NetCDF format
    dataset.to_netcdf(os.path.join(temp_dir, "data.nc"))

    # Save the dataset to HDF5 format.
    # Note that we use the same method as the netCDF format
    # This is because netCDF files are also valid HDF5 files.
    dataset.to_netcdf(os.path.join(temp_dir, "data.h5"))

    # Convert the dataset to a pandas DataFrame
    df = dataset.to_dataframe()

    # Save the DataFrame as CSV format
    df.to_csv(os.path.join(temp_dir, "data.csv"))

## Specifying Time-Dependent Parameters with Interpolation
Well ain't that nifty?

But what if manually writing out times and stuff is a bit tedious? What if you want to have some other code that generates a sequence of times and values and you want to use that instead? Sure, why not. One go-to place is the `popsim.interp` module which we use below to specify a current ramp trajectory.

In [ ]:
from popsim.interp import interp

# Specify a ramp rate and create an array of plasma currents.
ramp_rate = -0.5e6
current_trajectory = 8.7e6 + ramp_rate * time_base

# Interpolate and apply the interpolated trajectory to the params struct.
new_params.plasma_current = interp(time_base, current_trajectory)

sim_input = SimInput(time=time_base, initial_state=state, params=new_params)
# Simulate the new trajectory.
dataset = simulate(
    module=model,
    sim_inputs=sim_input,
)
visualize_time_series(dataset[visualize_vars])

## Running Batches of Simulations

Okay, running one simulation is good, but running a gazillion is great! To run multiple simulations with different settings, you can provide a list `SimInput` objects.


Imagine you want to compare a couple of cases:
   1) Baseline
   2) Impurity injection of tungsten
   3) Impurity spike of argon
   4) ICRF coupling efficiency drops suddenly

Well, we can run these different simulation cases, no problem. 

In [ ]:
tungsten_impurity_spike = {
    1.99: 0.0,
    2.0: 0.1,
    2.1: 0.1,
    2.11: 0.0,
}
argon_impurity_spike = {
    1.99: 0.0,
    2.0: 0.1,
    2.1: 0.1,
    2.11: 0.0,
}

icrf_drop = {
    1.99: 0.9,
    2.0: 0.5,
}

# Define baseline case as equivalent to the initial params.
baseline_case = SimInput(time=time_base, initial_state=state, params=params)

# Define the tungsten case.
tungsten_case = deepcopy(baseline_case)
tungsten_case.params.fueling19[Impurity.Tungsten] = tungsten_impurity_spike

# Define the argon case.
argon_case = deepcopy(baseline_case)
argon_case.params.fueling19[Impurity.Argon] = argon_impurity_spike

# Define the ICRF drop case.
icrf_drop_case = deepcopy(baseline_case)
icrf_drop_case.params.fraction_of_external_power_coupled = icrf_drop

cases = [
    baseline_case,
    tungsten_case,
    argon_case,
    icrf_drop_case,
]

dataset = simulate(
    module=model,
    sim_inputs=cases,
)
visualize_vars = [
    "params.fueling19.Impurity.Tungsten",
    "params.fueling19.Impurity.Argon",
    "params.fraction_of_external_power_coupled",
    "state.stored_energy",
    "state.density_state.vol_avg_ion.Impurity.Tungsten",
    "state.density_state.vol_avg_ion.Impurity.Argon",
    "state.hmode_state.hmode",
    "output.aux_data.Prad_imp_MW",
    "output.aux_data.tau_E",
]


visualize_time_series(dataset[visualize_vars])

## Generating Random Walks and Using MultiCase to Generate a Batch of Simulations

Particle transport faces a good amount of uncertainty. In the current `CometMirror` model, each species particle confinement time essentially treated as `k_{species}*tau_E`. Due to the uncertainty, it seems to make sense to treat these `k` parameters as random walks.

The code below geneates 10 random walks for the `particle_confinement_time_scalar` parameters. Here, we can actually use the `visualize_params` function to see the random walks.

In [ ]:
import jax

from popsim.stochastic import generate_random_walks
from popsim.visualize import visualize_params

# Diffusion mags specify the degree of randomness in the random walks.
diffusion_mags = {k: 0.25 for k in params.particle_confinement_scalar.keys()}
n_samps = 10

random_walks = generate_random_walks(
    jax.random.PRNGKey(42),  # Seed the random number generator.
    n_samps,
    time_base,
    params.particle_confinement_scalar,  # Specify a dictionary of inital conditions.
    diffusion_mags,  # Specify the degree of randomness in the random walks.
)
pprint(random_walks)
visualize_params(random_walks, time_base)

You may recall from earlier that `particle_confinement_scalar` is a dictionary in the 

```python
      particle_confinement_scalar={<FuelSpecies.Tritium: -3>: 3.0,
                                    <FuelSpecies.Deuterium: -2>: 3.0,
                                    <Impurity.Helium: 2>: 10.0,
                                    <Impurity.Oxygen: 8>: 10.0,
                                    <Impurity.Tungsten: 74>: 10.0},
```

But the random walk generator output a list of interpolations. But have no fear, we can call an interp function, and wrap it with `MultiCases` and construct our `SimInput` as usual.

`MultiCases` essentially specifies that the list provided defines a single simulation case each. You can have multiple `MultiCases` objects, and the semantics are that the first element of each `MultiCase` corresponds to the first element of the other `MultiCase`, and so on.

For the sake of demonstration, let's generate 10 different auxiliary heating cases as well; one for each random walk.

In [ ]:
new_params = deepcopy(params)


# Apply the random walks to the params struct.
new_params.particle_confinement_scalar = MultiCases(cases=random_walks)

# Also vary heating across the cases. Note that we need to make sure all MultiCases have the same length.
heating_scales = jnp.linspace(0.8, 1.2, len(random_walks))
new_params.P_aux_MW = MultiCases(cases=[new_params.P_aux_MW * scale for scale in heating_scales])

sim_inputs = SimInput(time=time_base, initial_state=state, params=new_params)

When a `SimInput` instance has `MultiCases` objects, you can call `generate_sim_cases` and you'll get back a list of `SimInput` objects for each case.


In [ ]:
# Use generate_sim_cases to generate all possible combinations of the MultiCases.
sim_inputs_list = sim_inputs.generate_sim_cases()
print(len(sim_inputs_list))

Now we can run the simulation cases as usual and visualize the results.

In [ ]:
dataset = simulate(
    module=model,
    sim_inputs=sim_inputs_list,
)
visualize_vars = [
    "params.P_aux_MW",
    "state.stored_energy",
    "state.density_state.vol_avg_ion.Impurity.Tungsten",
    "state.density_state.vol_avg_ion.FuelSpecies.Deuterium",
    "state.density_state.vol_avg_ion.FuelSpecies.Tritium",
]
visualize_time_series(dataset[visualize_vars])

## Generating Batches of Simulations with CombinatorialCases

Okay, great! So `MultiCases` allows us to specify multiple different scenarios. But sometimes, we want to generate all possible combinations of scenarios. For example, perhaps we want to scan auxiliary heating rates for every random walk trajectory we saw above. This is where `CombinatorialCases` comes in.

When you specify `CombinatorialCases`, each instance of it gets combined with every other. So in the example below, the number of simulations run will be the product of the number of elements in each list. As an example, let's consider combinatorially generate simulations for:

    1. Different particle confinement random walks
    2. Different auxiliary heating rates
    3. Different impurity spike scenarios
    4. Nominal and off-nominal ICRF coupling efficiencies

We would expect to see 10 * 4 * 4 * 2 = 320 simulations run.

In [ ]:
new_params = deepcopy(params)

new_params.particle_confinement_scalar = CombinatorialCases(cases=random_walks)
new_params.P_aux_MW = CombinatorialCases(cases=[9.0, 10.0, 11.0, 12.0])
new_params.fueling19[Impurity.Tungsten] = CombinatorialCases(cases=[0.0, tungsten_impurity_spike])
new_params.fueling19[Impurity.Argon] = CombinatorialCases(cases=[0.0, argon_impurity_spike])
new_params.fraction_of_external_power_coupled = CombinatorialCases(cases=[0.9, icrf_drop])

sim_inputs = SimInput(time=time_base, initial_state=state, params=new_params)

# Use generate_sim_cases to generate all possible combinations of the CombinatorialCases.
sim_inputs_list = sim_inputs.generate_sim_cases()

dataset = simulate(
    module=model,
    sim_inputs=sim_inputs_list,
)
dataset

## POPSIM GUI
You can also input the simulation output datasets into a GUI.

## POPSIM GUI
There is also an interactive GUI you can use to visualize the simulation results. Try it out by running the cell below!

In [ ]:
import panel as pn

from popsim.gui import PopsimGUI

gui = PopsimGUI(dataset, time_dim="time", rho_dim="rho", simulation_dim="simulation")

# Uncomment to try out the gui.
# pn.serve(gui.build_view())